# Imports

In [25]:
# Source - https://stackoverflow.com/a
# Posted by bruno-uy
# Retrieved 2025-11-09, License - CC BY-SA 4.0
!jupyter nbextension enable --py widgetsnbextension

Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [26]:
import torch
import torch.nn as nn
import matplotlib
import time
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import MPNetModel, MPNetConfig
import numpy
import pandas as pd
from dataclasses import dataclass, replace
from typing import Dict, Optional, Tuple

In [27]:
torch.cuda.is_available()

False

In [28]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [29]:
print(device)

cpu


In [30]:
# seed = 42
# random.seed(seed)
# torch.manual_seed(seed)
# env.reset(seed=seed)
# env.action_space.seed(seed)
# env.observation_space.seed(seed)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed(seed)

# Utils

In [31]:
def get_all_combos(n: int, k:int, device: torch.device) -> torch.LongTensor:
    idx = torch.arange(n, device=device)
    # this returns combinations in sorted order
    return torch.combinations(idx, r=k)

# Model Classes

In [32]:
class ConnectionsDataset(torch.utils.data.Dataset):
    def __getitem__(self, idx):
        raise NotImplementedError
    def __len__(self):
        raise NotImplementedError

In [33]:
class Embedder:
    """
    words: list of N strings
    return: (N, D) tensor
    """
    def encode(self, words: list[str]) -> torch.Tensor:
        raise NotImplementedError

In [34]:
class Contextualizer:
    def __call__(self, emb: torch.Tensor) -> torch.Tensor:
        """
        emb: (N, D)
        return: (N, D) contextualized embeddings
        """
        raise NotImplementedError

In [35]:
class Grouper:
    """
    emb: (N, D)
    return (1820, N, D) outputs all possible groups of 4 of the embeddings
    """

    def group(self, n: int, embeddings: torch.tensor) -> torch.Tensor:
        raise NotImplementedError

In [36]:
class ActionScorer:
    def q_values(self, state_emb: torch.tensor) -> torch.Tensor:
        """
        state_emb: (N, D)
        return: (A, ) where A is the number of possible actions (1820 for 4 groupings out of 16)
        """
        raise NotImplementedError

In [37]:
class Model:
    def select_action(self, q_values: torch.Tensor, mask=None) -> int:
        """
        q_values: (A, )
        mask: (A, ) bolean or None
        return: int (choose action index)
        """
        raise NotImplementedError

    def train_step(self, batch):
        raise NotImplementedError

# State Classes

In [38]:
@dataclass(frozen=True)
class GameConfig:
    R1_win: float = 10.0
    R2_correct: float = 1.0
    R3_one_away: float = 0.2
    R4_wrong: float = -0.5
    R5_game_over: float = -2.0
    init_lives: int = 4

In [39]:
@dataclass(frozen=True)
class BoardTensors:
    words: torch.Tensor # (16, D) float
    group_labels: torch.LongTensor # (16,) int in {0, 1, 2, 3}
    combos: torch.LongTensor # (1820, 4) long

In [40]:
@dataclass(frozen=True)
class GameState:
    words_mask: torch.BoolTensor # (16,) - words that we have remaining
    found_groups: torch.BoolTensor # (4,) - groups that we have already found
    lives: torch.Tensor # () int32 - remaining lives
    actions_mask: torch.BoolTensor # (1820,) - actions that we used

    @staticmethod
    def game_start(board: BoardTensors, config: GameConfig) -> "GameState":
        device = board.words.device

        words = torch.ones(16, dtype=torch.bool, device=device)
        groups = torch.zeros(4, dtype=torch.bool, device=device)
        lives = torch.tensor(config.init_lives, dtype=torch.long, device=device)
        actions = torch.ones(board.combos.size(0), dtype=torch.bool, device=device)

        return GameState(
            words_mask = words,
            found_groups = groups,
            lives = lives,
            actions_mask = actions,
        )


In [41]:
class Game:

    def __init__(self, config: Optional[GameConfig] = None):
        self.config = config or GameConfig()

    @torch.no_grad()
    def make_guess(self,
                   board: BoardTensors,
                   state: GameState,
                   action_id: int
    ) -> Tuple[GameState, torch.Tensor, bool, Dict]:
        """
        Function that updates the states after the agent
        makes a guess for a potential group.

        Returns: (next_state, reward, is_finished, logging)
        """
        device = board.words.device
        finished = False
        logs = {}

        # Get the group of words that we guessed
        combo = board.combos[action_id]

        # Check that all words picked by action are live
        if not state.words_mask.index_select(0, combo).all():

            # Penalize the illegal action same as a wrong guess, but
            # don't decrease live count
            reward = torch.tensor(self.config.R4_wrong, device=device)

            return (state, reward, False, {"result": "illegal"})

        # Get the category labels for each word
        labels = board.group_labels.index_select(0, combo) # (4,)

        # Get the number of words that are in each label
        group_counts = torch.bincount(labels, minlength=4) # (4,)

        max_count, group = group_counts.max(0)
        group_idx = int(group.item())

        reward_val = 0
        logging = {}
        # Correct guess (all 4 words in a label and we haven't discovered the group yet)
        if max_count.item() == 4:
            reward_val = reward_val + self.config.R2_correct
            # Update the mask for the remaining words
            new_words_mask = state.words_mask.clone()
            new_words_mask[combo] = False

            # Update the mask for the found groups
            new_found_groups = state.found_groups.clone()
            new_found_groups[group_idx] = True

            # Update the action mask to remove actions that use words that we already used
            # and the action we took at this step
            new_actions_mask = self._remove_used_action(board, state, action_id)
            no_used_words_mask = self._remove_actions_with_used_words(board, state, new_actions_mask)

            new_actions_mask &= no_used_words_mask

            # If we found all groups this means we won
            if new_found_groups.all():
                reward_val = reward_val + self.config.R1_win
                finished = True

            next_state = replace(
                state,
                words_mask = new_words_mask,
                found_groups = new_found_groups,
                actions_mask = new_actions_mask,
            )
            logging = {"result": "correct", "category": group_idx}


        # If we are one-away
        elif max_count.item() == 3:
            reward_val = self.config.R3_one_away + self.config.R4_wrong
            reward = torch.tensor(self.config.R3_one_away, device=device)
            new_lives = state.lives - 1
            new_actions_mask = self._remove_used_action(board, state, action_id)

            next_state = replace(
                state,
                lives=new_lives,
                actions_mask=new_actions_mask,
            )

        # If we made the wrong guess
        else:
            reward_val = self.config.R4_wrong
            new_lives = state.lives - 1
            game_over = new_lives.item() <= 0
            new_actions_mask = self._remove_used_action(board, state, action_id)

            next_state = replace(
                state,
                lives = new_lives,
                actions_mask=new_actions_mask,
            )


        # If game over we get a game_over reward/penalty
        if new_lives.item() <= 0:
            reward_val = reward_val +  self.config.R5_game_over
            finished = True

        # return the state update
        reward = torch.tensor(reward_val, device=device)
        return (next_state, reward, finished, logging)


    def _remove_used_action(self,
                            board: BoardTensors,
                            state: GameState,
                            picked_action: int
                           ) -> torch.BoolTensor:
        new_actions_mask = state.actions_mask.clone()
        new_actions_mask[picked_action] = False
        return new_actions_mask

    def _remove_actions_with_used_words(self,
                            board: BoardTensors,
                            state: GameState,
                            new_words_mask: torch.BoolTensor
                           ) -> torch.BoolTensor:
        # Get all the possible word combos
        combos = board.combos

        # Create a mask that zeros out combos that have words that were already used
        no_used_words = new_words_mask[combos].all(dim=1)

        return no_used_words



# Architecture Specific Classes

In [42]:
class MiniLMEmbedding(Embedder):

    def __init__(self):
        self.model = model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    def encode(self, words: list[str], device) -> torch.Tensor:
        return self.model.encode(
            words,
            convert_to_tensor=True,
            device=str(device)
        )

In [43]:
class TransformerEncoderContextualizer(nn.Module):

    def __init__(self, d_size=384, n_layers=4, n_head=8, ff_dim=1024):
        super().__init__()
        self.encoder_layer = nn.TranformerEncoderLayer(
            d_size=d_size,
            nhead=n_layers,
            dim_feedforward=ff_dim,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, word_embeddings):
        """
        Input: (B, 16, 384)
        Returns (B, 16, 384)
        """
        return self.encoder(word_embeddings)


In [44]:
class SimpleGrouper(Grouper):

    def __init__(self, device):
        self.device = device

    def group(self, n:int, embeddings: torch.Tensor) -> torch.Tensor:
        """
        Input: (B, 16, 384)
        Output: (B, 1820, 4, 384)
        """
        combos = get_all_combos(n, 4, self.device)
        # Reshape combos so they are batch-friendly
        combos = combos.unsqueeze(0).expand(B, -1, -1)
        return embeddings.gather(1, combos.unsqueeze(-1).expand(-1, -1, -1, D))

In [45]:
class RelationNetworkScorer(nn.Module):

    def __init__(self,
                 device,
                 d_size=384,
                 g_hidden=256,
                 f_hidden=128,
                 output_size=1):
        """
        Inspired by the code here:
        https://diegovianagomes.medium.com/lets-develop-a-simple-neural-network-module-for-relational-reasoning-part-1-97762ca22e01
        """
        super().__init__()

        # This is g_theta from the paper
        self.g = nn.Sequentual(
            nn.Linear(2 * d, g_hidden),
            nn.ReLU(),
            nn.Linear(g_hidden, g_hidden),
            nn.ReLU()
        )

        # This is f_phi from the paper
        self.f = nn.Sequential(
            nn.Linear(g_hidden, f_hidden),
            nn.ReLU(),
            nn.Linear(f_hidden, output_size)

        )
        self.device = device

        # Get all pairwaise indexes possible of a group of 4
        self.pair_combos = get_all_combos(4, 2, self.device)

    def forward(self, group_embeddings):
        """
        emb: (Batch, 4, 384)
        returns (B,) score
        """
        group_pairs = group_embeddings[:, self.pair_combos]

        # Concatenate the embedding pairs, (B, 6, 2*384)
        group_pairs = group_pairs.reshape(group_embeddings.shape[0],
                                         -1, 2 * group_embeddings.shape[2])

        relations = self.g(group_pairs)

        # Sum over the relations, (B, g_hidden)
        relations_sum = relations.sum(dim=1)

        # Get the scores
        scores = self.f(relations_sum).squeeze(-1)

        return scores


# DQN w/ Replay

In [46]:
class ReplayMemory:
    """
    Class so that the DQN can store the transitions it encountered.
    This code was inspired by the following PyTorch article:
    https://docs.pytorch.org/tutorials/intermediate/reinforcement_q_learning.html
    """

    def __init__(self, capacity, device):
        self.cap = capacity
        self.device = device
        self.idx = 0
        self.full = False

        # Transitions
        self.s, self.a, self.r, self.sp = None, None, None, None

    def push(self, s, a, r, sp):
        """
        Store the transitions in a queue
        """
        if self.states is None:
            self.s = torch.zeros((self.capacity, *s.shape), device=self.device, dtype=s.dtype)
            self.a = torch.zeros((self.capacity, *a.shape), device=self.device, dtype=a.dtype)
            self.r = torch.zeros((self.capacity, *r.shape), device=self.device, dtype=r.dtype)
            self.sp = torch.zeros((self.capacity, *sp.shape), device=self.device, dtype=sp.dtype)

        self.s[self.idx] = s
        self.a[self.idx] = a
        self.r[self.idx] = r
        self.sp[self.idx] = sp

        self.idx = (self.idx + 1) % self.cap
        if not self.idx:
            self.full = True

    def sample(self, batch_size):
        max_idx = self.cap if self.full else self.idx
        sample_idx = torch.randint(0, max_index, (batch_size,), device=self.device)

        return (
            self.s[sample_idx],
            self.a[sample_idx],
            self.r[sample_idx],
            self.sp[sample_idx]
        )

    def __len__(self):
        return self.cap if self.full else self.idx


In [47]:
class DQN(nn.Module):

    def __init__(self, n_obs, n_actions, n_hidden=128):
        super(DQN, self).__init__()

        self.q = nn.Sequentual(
                    nn.Linear(n_obs, n_hidden),
                    nn.ReLU(),
                    nn.Linear(n_hidden, n_hidden),
                    nn.ReLU(),
                    nn.Linear(n_hidden, n_actions),
                )

    def forward(self, observations):
        return self.q(observations)

# Data Exploration/Cleaning

In [48]:
# Getting the CSV into a df
df = pd.read_csv("Connections_Data.csv")
df.head()

,Game ID,Puzzle Date,Word,Group Name,Group Level,Starting Row,Starting Column
0,1,2023-06-12,SNOW,WET WEATHER,0,1,1
1,1,2023-06-12,LEVEL,PALINDROMES,3,1,2
2,1,2023-06-12,SHIFT,KEYBOARD KEYS,2,1,3
3,1,2023-06-12,KAYAK,PALINDROMES,3,1,4
4,1,2023-06-12,HEAT,NBA TEAMS,1,2,1


In [49]:
# We have 880 games
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14224 entries, 0 to 14223
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Game ID          14224 non-null  int64 
 1   Puzzle Date      14224 non-null  object
 2   Word             14222 non-null  object
 3   Group Name       14224 non-null  object
 4   Group Level      14224 non-null  int64 
 5   Starting Row     14224 non-null  int64 
 6   Starting Column  14224 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 778.0+ KB


In [50]:
df[df["Word"].isna()]

,Game ID,Puzzle Date,Word,Group Name,Group Level,Starting Row,Starting Column
930,59,2023-08-09,NaN,UNSPECIFIED QUANTITIES,0,1,3
978,62,2023-08-12,NaN,PERIODIC TABLE SYMBOLS,3,1,3


In [51]:
# Remove games where some of the words are NULL
bad_ids = [59, 62]
df.drop(df[df["Game ID"].isin(bad_ids)].index, inplace=True)

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14192 entries, 0 to 14223
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Game ID          14192 non-null  int64 
 1   Puzzle Date      14192 non-null  object
 2   Word             14192 non-null  object
 3   Group Name       14192 non-null  object
 4   Group Level      14192 non-null  int64 
 5   Starting Row     14192 non-null  int64 
 6   Starting Column  14192 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 887.0+ KB


In [53]:
df = df.sort_values(
    by=["Game ID", "Group Level", "Word"]
).reset_index(drop=True)
df.head()

,Game ID,Puzzle Date,Word,Group Name,Group Level,Starting Row,Starting Column
0,1,2023-06-12,HAIL,WET WEATHER,0,3,2
1,1,2023-06-12,RAIN,WET WEATHER,0,3,4
2,1,2023-06-12,SLEET,WET WEATHER,0,4,1
3,1,2023-06-12,SNOW,WET WEATHER,0,1,1
4,1,2023-06-12,BUCKS,NBA TEAMS,1,2,3


# Experiment

In [54]:
class Experiment:

    # Set these params to None initially for initial validation of the class, remember to remove these defaults later
    # Also, not sure we need env yet.
    def __init__(self,
                 df,
                 word_cnt,
                 embedder: Embedder,
                 device,
                 batch_size=16,
                 grouper: Grouper | None = None,
                 contextualizer: nn.Module | None = None,
                 scorer: nn.Module | None = None,
                 agent=None,
                 env=None,
                 embeddings=None):
        self.df = df
        self.n = word_cnt
        self.embedder = embedder
        self.grouper = grouper
        self.device = device
        self.B = batch_size
        self.contextualizer = contextualizer
        self.scorer = scorer
        self.agent = agent
        self.env = env
        self.embeddings = None
        self.board_tensors = []

    def get_all_word_embeddings(self):
        """
        1. Step
        Helper to get all word embeddings
        Returns: (num_games * 16, 384)
        """
        if not self.embeddings:
            embeddings = self.embedder.encode(self.df["Word"].tolist(), self.device)
            torch.save(embeddings, f"{self.embeddings.__class__.__name__}_embeddings.pt")
            self.embeddings = embeddings

    def _contextualize(self, batched_embs: torch.tensor) -> torch.tensor:
        """
        2nd Step
        For each group of 16 words contextualize the embeddings
        Returns: (B, 16, 384)
        """
        outputs = []
        for batch in batched_embs:
            ctx_embeddings = self.contextualizer.forward(batch)
            outputs.append(ctx_embeddings)
        return ctx_embeddings

    def _group_embeddings(self, batched_embs: torch.tensor) -> torch.tensor:
        """
        3rd Step
        For each board we output all possible groups of words
        Returns: (B, 1820, 4, 384)
        """
        outputs = []
        for batch in batched_embs:
            grouped_embs = self.grouper.group(batch)
            outputs.append(grouped_embs)
        return outputs

    def _score_groups(self, group_embeddings: torch.tensor) -> torch.tensor:
        """
        4th Step
        """
        output = []
        for combos in group_embeddings:
            scores = self.scorer.forward(combos)
            output.append(scores)
        return output

    def _batch_embeddings(self, embeddings):
        batched_embeddings = embeddings.clone()
        return batched_embeddings.reshape(self.B, self.n, 384)

    def get_word_embeddings(self, game_id):
        """
        Helper to get embeddings for a single game
        """
        embeddings = self.embedder.encode(self.df[self.df["Game ID"] == game_id]["Word"].tolist(), self.device)
        return embeddings

    def all_df_to_game_states(self):
        """
        Puts each board from the df into a BoardTensors class,
        which takes in the 16 words as torch embeddings, their
        category tensor, and all the actions you can do with
        this board.
        """
        self.get_all_word_embeddings() # (16 * number of games, 384)

        # Go through each 16 words in a board
        for game_id, game_df in self.df.groupby("Game ID"):
            game_embeddings = embeddings[idx:idx + 16]
            # Since we presorted our df
            categories = torch.arange(4, device=self.device).repeat_interleave(4) # (16,)
            combos = get_all_combos(self.n, 4, self.device)
            board_tensors = BoardTensors(game_embeddings, categories, combos)
            self.board_tensors.append(board_tensors)
            idx += 16
        return self.board_tensors

    def df_to_game(self, game_id):
        """
        Function to put a single board into a BoardTensor.
        """
        game_embeddings = self.get_word_embeddings(game_id)
        categories = torch.arange(4, device=self.device).repeat_interleave(4)
        combos = get_all_combos(self.n, 4, self.device)
        board_tensors = BoardTensors(game_embeddings, categories, combos)
        return board_tensors


# Logic Validation

In [55]:
df.head(16)

,Game ID,Puzzle Date,Word,Group Name,Group Level,Starting Row,Starting Column
0,1,2023-06-12,HAIL,WET WEATHER,0,3,2
1,1,2023-06-12,RAIN,WET WEATHER,0,3,4
2,1,2023-06-12,SLEET,WET WEATHER,0,4,1
3,1,2023-06-12,SNOW,WET WEATHER,0,1,1
4,1,2023-06-12,BUCKS,NBA TEAMS,1,2,3
5,1,2023-06-12,HEAT,NBA TEAMS,1,2,1
6,1,2023-06-12,JAZZ,NBA TEAMS,1,3,1
7,1,2023-06-12,NETS,NBA TEAMS,1,4,4
8,1,2023-06-12,OPTION,KEYBOARD KEYS,2,3,3
9,1,2023-06-12,RETURN,KEYBOARD KEYS,2,2,4


In [56]:
test_exp = Experiment(
    df=df,
    word_cnt=16,
    embedder=MiniLMEmbedding(),
    grouper=SimpleGrouper(device),
    device=device,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [57]:
test_game = test_exp.df_to_game(1)
test_game

BoardTensors(words=tensor([[-0.1149,  0.1230,  0.0414,  ..., -0.0377,  0.0363,  0.0759],
        [-0.0259,  0.0191,  0.0624,  ..., -0.0290, -0.0521,  0.0364],
        [-0.0522,  0.0074, -0.0133,  ..., -0.0651, -0.0631,  0.0272],
        ...,
        [-0.0815, -0.0077, -0.0857,  ...,  0.0268, -0.0626, -0.0200],
        [-0.0583,  0.0731,  0.0069,  ..., -0.0102,  0.0923,  0.0631],
        [ 0.0027,  0.1290,  0.0088,  ..., -0.0235,  0.0245,  0.0061]]), group_labels=tensor([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3]), combos=tensor([[ 0,  1,  2,  3],
        [ 0,  1,  2,  4],
        [ 0,  1,  2,  5],
        ...,
        [11, 12, 14, 15],
        [11, 13, 14, 15],
        [12, 13, 14, 15]]))

In [58]:
test_game_config = GameConfig()
test_game_config

GameConfig(R1_win=10.0, R2_correct=1.0, R3_one_away=0.2, R4_wrong=-0.5, R5_game_over=-2.0, init_lives=4)

In [59]:
test_game_state = GameState.game_start(test_game, test_game_config)
test_game_state

GameState(words_mask=tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True]), found_groups=tensor([False, False, False, False]), lives=tensor(4), actions_mask=tensor([True, True, True,  ..., True, True, True]))

In [60]:
test_game_env = Game()

In [61]:
# This should count as a correct guess
test_game_env.make_guess(test_game, test_game_state, 30)

(GameState(words_mask=tensor([True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True]), found_groups=tensor([False, False, False, False]), lives=tensor(3), actions_mask=tensor([True, True, True,  ..., True, True, True])),
 tensor(-0.5000),
 False,
 {})

# Experiment